# 09 - Prepare RecBole Data for Sequential Models

This notebook converts ReDial dialogue data to RecBole **pre-augmented sequential format**
for sequential models (SASRec, GRU4Rec, NARM, SRGNN).

**Why pre-augmentation is needed:**
RecBole bug [#1593](https://github.com/RUCAIBox/RecBole/issues/1593): When `benchmark_filename` is set,
`SequentialDataset._change_feat_format()` skips `data_augmentation()`. The workaround is to
pre-augment the data ourselves with sliding window sequences.

**Sliding window augmentation:**
For a user with liked items [L1, L2, L3] and accepted items [A1, A2]:
- Full sequence: [L1, L2, L3, A1, A2]
- Training rows: `[L1]->L2`, `[L1,L2]->L3`, `[L1,L2,L3]->A1`, `[L1,L2,L3,A1]->A2`

**Split strategy (mirrors notebook 06):**
- Same 80/20 train/valid split (`random_state=42`)
- Same title-to-ID mapping
- **Warm-start**: liked→liked transitions go to train.inter for ALL users (including valid/test),
  matching notebook 06's approach so RecBole can mask liked items during evaluation
- Train targets: full sliding window for accepted items
- Valid/Test targets: **frozen history** (liked items only) for fair LLM comparison

**Outputs:**
- `data/recbole/redial_seq/redial_seq.train.inter`
- `data/recbole/redial_seq/redial_seq.valid.inter`
- `data/recbole/redial_seq/redial_seq.test.inter`
- `data/recbole/redial_seq/redial_seq.item`
- `data/recbole/redial_seq/evaluation_targets.json`
- `data/recbole/redial_seq/id_to_title.json`

In [ ]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Configuration - MUST match notebook 06 exactly
DATA_PATH = Path("../data")
RECBOLE_SEQ_PATH = DATA_PATH / "recbole" / "redial_seq"
SEED = 42
VALID_SPLIT = 0.2
MAX_ITEM_LIST_LENGTH = 50
N_PERMUTATIONS = (
    4  # Random permutations of liked items for train augmentation (0 = disabled)
)

## Load Source Data

In [ ]:
# Load movie catalog
movies_df = pd.read_csv(DATA_PATH / "processed" / "movies_with_mentions_processed.csv")
print(f"Loaded {len(movies_df)} movies")
movies_df.head()

In [ ]:
# Load train dialogues
train_dialogues = []
with open(DATA_PATH / "processed" / "train_prompt_templates.jsonl") as f:
    for line in f:
        train_dialogues.append(json.loads(line))
print(f"Loaded {len(train_dialogues)} train dialogues")

# Load test dialogues
test_dialogues = []
with open(DATA_PATH / "processed" / "test_prompt_templates.jsonl") as f:
    for line in f:
        test_dialogues.append(json.loads(line))
print(f"Loaded {len(test_dialogues)} test dialogues")

## Create Title to ID Mapping

In [ ]:
# Create movie title to ID mapping (use both title and title_norm)
title_to_id = {}
for _, row in movies_df.iterrows():
    title_to_id[row["title"]] = row["id"]
    title_to_id[row["title_norm"]] = row["id"]

# Create ID to title mapping for later use
id_to_title = dict(zip(movies_df["id"], movies_df["title_norm"]))

print(f"Created mapping for {len(movies_df)} movies")
print(f"Total title variants: {len(title_to_id)}")

In [ ]:
def match_title_to_id(title: str, title_to_id: dict[str, int]) -> int | None:
    """Match a movie title to its ID with fuzzy matching fallback."""
    # Direct match
    if title in title_to_id:
        return title_to_id[title]

    # Normalized match (lowercase, stripped)
    title_lower = title.strip().lower()
    for key, val in title_to_id.items():
        if key.lower() == title_lower:
            return val

    return None

## Split Train Data into Train/Valid

In [ ]:
# MUST use same split as notebook 06
train_dialogues_split, valid_dialogues = train_test_split(
    train_dialogues,
    test_size=VALID_SPLIT,
    random_state=SEED,
)

print(f"Train dialogues: {len(train_dialogues_split)}")
print(f"Valid dialogues: {len(valid_dialogues)}")
print(f"Test dialogues:  {len(test_dialogues)}")

## Sequential Data Augmentation

Sliding window approach:
- For each dialogue, construct full sequence: `[liked_items..., accepted_items...]`
- Apply sliding window: for position `i` (from 1 to len-1), create row:
  - `user_id`: user ID
  - `item_id`: `sequence[i]` (target)
  - `item_id_list`: `sequence[0:i]` (history), space-separated, truncated to `MAX_ITEM_LIST_LENGTH`

**Train split:** Include ALL sliding window pairs (both liked→liked and liked→accepted transitions)  
**Valid/Test splits:** Only include pairs where the TARGET is an accepted item

**Masking warm-start (mirrors notebook 06):**
For ALL users (including valid/test), liked→liked transitions go into train.inter.
This ensures RecBole can mask liked items during full-set evaluation, matching
the CF setup in notebook 06.

**Frozen history for valid/test:**
To make the comparison fair with LLMs (which only see liked items as context),
valid/test target predictions use a frozen history equal to the liked items only.
No sliding window that leaks accepted items into the history.

In [ ]:
def prepare_sequential_splits(
    train_dialogues: list[dict],
    valid_dialogues: list[dict],
    test_dialogues: list[dict],
    title_to_id: dict[str, int],
    max_seq_len: int = MAX_ITEM_LIST_LENGTH,
    n_permutations: int = 0,
    seed: int = 42,
) -> dict:
    """Create pre-augmented sequential interaction splits.

    For each dialogue:
      1. Build full sequence: [liked_item_ids..., accepted_item_ids...]
      2. Warm-start: liked→liked transitions go to train.inter for ALL users
         (mirrors notebook 06 so RecBole can mask liked items during eval)
      3. Train targets: full sliding window for accepted items
      4. Valid/Test targets: frozen history (only liked items) for fair LLM comparison

    User IDs are assigned with a global counter across all splits
    (same iteration order as notebook 06).
    """
    train_rows = []
    valid_rows = []
    test_rows = []

    train_targets = {}
    valid_targets = {}
    test_targets = {}

    train_user_ids = []
    valid_user_ids = []
    test_user_ids = []

    unmatched_titles = set()
    rng = np.random.default_rng(seed)
    perm_rows_added = 0

    skipped_no_liked = 0
    skipped_no_accepted = 0
    skipped_too_short = 0

    def _resolve_ids(titles: list[str]) -> list[int]:
        ids = []
        for t in titles:
            item_id = match_title_to_id(t, title_to_id)
            if item_id is not None:
                ids.append(item_id)
            else:
                unmatched_titles.add(t)
        return ids

    user_id = 0  # Global counter (same as notebook 06)

    for split_name, dialogues in [
        ("train", train_dialogues),
        ("valid", valid_dialogues),
        ("test", test_dialogues),
    ]:
        for dialogue in tqdm(dialogues, desc=f"Processing {split_name}"):
            liked_ids = _resolve_ids(dialogue.get("user_liked", []))
            accepted_ids = _resolve_ids(dialogue.get("recommended_accepted", []))

            # Skip dialogues with no liked items (can't form any sequence)
            if not liked_ids:
                skipped_no_liked += 1
                user_id += 1
                continue

            # Skip dialogues with no accepted items (no prediction targets)
            if not accepted_ids:
                skipped_no_accepted += 1
                user_id += 1
                continue

            # Need at least 2 items total to form one sequence pair
            full_sequence = liked_ids + accepted_ids
            if len(full_sequence) < 2:
                skipped_too_short += 1
                user_id += 1
                continue

            n_liked = len(liked_ids)

            # --- Warm-start: liked→liked transitions go to train for ALL users ---
            # This mirrors notebook 06 where ALL users' liked items are in train.inter,
            # ensuring RecBole can mask them during full-set evaluation.
            for i in range(1, n_liked):
                target_item = liked_ids[i]
                history = liked_ids[max(0, i - max_seq_len) : i]
                train_rows.append(
                    {
                        "user_id": user_id,
                        "item_id": target_item,
                        "item_id_list": " ".join(str(x) for x in history),
                    }
                )

            # Anchor row for first liked item (so it also gets masked)
            if n_liked >= 1:
                train_rows.append(
                    {
                        "user_id": user_id,
                        "item_id": liked_ids[0],
                        "item_id_list": "0",
                    }
                )

            # --- Target predictions (accepted items) ---
            # For valid/test: freeze history at liked items only (matches LLM context).
            # For train: use full sliding window for maximum training signal.
            frozen_history = liked_ids[-max_seq_len:]
            frozen_history_str = " ".join(str(x) for x in frozen_history)

            for i in range(n_liked, len(full_sequence)):
                target_item = full_sequence[i]

                if split_name == "train":
                    # Train: sliding window gives more training data
                    history = full_sequence[max(0, i - max_seq_len) : i]
                    train_rows.append(
                        {
                            "user_id": user_id,
                            "item_id": target_item,
                            "item_id_list": " ".join(str(x) for x in history),
                        }
                    )

                elif split_name == "valid":
                    valid_rows.append(
                        {
                            "user_id": user_id,
                            "item_id": target_item,
                            "item_id_list": frozen_history_str,
                        }
                    )
                else:  # test
                    test_rows.append(
                        {
                            "user_id": user_id,
                            "item_id": target_item,
                            "item_id_list": frozen_history_str,
                        }
                    )

            # --- Permutation augmentation (train only) ---
            # The user_liked order is arbitrary (extraction order from dialogue).
            # Permuting liked items teaches the model that the SET matters, not order.
            if split_name == "train" and n_permutations > 0 and n_liked >= 2:
                max_unique = math.factorial(min(n_liked, 10)) - 1
                n_perms = min(n_permutations, max_unique)
                seen = {tuple(liked_ids)}
                generated = 0
                attempts = 0
                while generated < n_perms and attempts < n_perms * 10:
                    attempts += 1
                    perm_liked = list(rng.permutation(liked_ids))
                    perm_key = tuple(perm_liked)
                    if perm_key in seen:
                        continue
                    seen.add(perm_key)
                    generated += 1

                    perm_sequence = perm_liked + accepted_ids

                    # Anchor row
                    train_rows.append(
                        {
                            "user_id": user_id,
                            "item_id": perm_liked[0],
                            "item_id_list": "0",
                        }
                    )
                    # Sliding window for permuted liked items
                    for pi in range(1, len(perm_liked)):
                        ph = perm_liked[max(0, pi - max_seq_len) : pi]
                        train_rows.append(
                            {
                                "user_id": user_id,
                                "item_id": perm_liked[pi],
                                "item_id_list": " ".join(str(x) for x in ph),
                            }
                        )
                    # Sliding window for accepted items with permuted prefix
                    for pi in range(len(perm_liked), len(perm_sequence)):
                        ph = perm_sequence[max(0, pi - max_seq_len) : pi]
                        train_rows.append(
                            {
                                "user_id": user_id,
                                "item_id": perm_sequence[pi],
                                "item_id_list": " ".join(str(x) for x in ph),
                            }
                        )
                    perm_rows_added += len(perm_liked) + len(accepted_ids)

            # Track targets and user IDs
            if split_name == "train":
                train_targets[user_id] = accepted_ids
                train_user_ids.append(user_id)
            elif split_name == "valid":
                valid_targets[user_id] = accepted_ids
                valid_user_ids.append(user_id)
            else:
                test_targets[user_id] = accepted_ids
                test_user_ids.append(user_id)

            user_id += 1

    print(f"Skipped {skipped_no_liked} dialogues with no liked items")
    print(f"Skipped {skipped_no_accepted} dialogues with no accepted items")
    print(f"Skipped {skipped_too_short} dialogues with < 2 items")
    if n_permutations > 0:
        print(
            f"Permutation augmentation: {perm_rows_added} additional train rows from {n_permutations} permutations"
        )
    print(f"Unmatched titles: {len(unmatched_titles)}")
    if unmatched_titles:
        print(f"Sample unmatched: {list(unmatched_titles)[:5]}")

    return {
        "train_df": pd.DataFrame(train_rows),
        "valid_df": pd.DataFrame(valid_rows),
        "test_df": pd.DataFrame(test_rows),
        "train_targets": train_targets,
        "valid_targets": valid_targets,
        "test_targets": test_targets,
        "train_user_ids": train_user_ids,
        "valid_user_ids": valid_user_ids,
        "test_user_ids": test_user_ids,
    }

In [ ]:
splits = prepare_sequential_splits(
    train_dialogues_split,
    valid_dialogues,
    test_dialogues,
    title_to_id,
    n_permutations=N_PERMUTATIONS,
    seed=SEED,
)

train_seq_df = splits["train_df"]
valid_seq_df = splits["valid_df"]
test_seq_df = splits["test_df"]
train_targets = splits["train_targets"]
valid_targets = splits["valid_targets"]
test_targets = splits["test_targets"]
train_user_ids = splits["train_user_ids"]
valid_user_ids = splits["valid_user_ids"]
test_user_ids = splits["test_user_ids"]

print("\nSplit summary:")
print(f"  Train .inter: {len(train_seq_df)} rows ({len(train_user_ids)} users)")
print(f"  Valid .inter: {len(valid_seq_df)} rows ({len(valid_user_ids)} users)")
print(f"  Test .inter:  {len(test_seq_df)} rows ({len(test_user_ids)} users)")

## Create RecBole Atomic Files

In [ ]:
def create_sequential_inter_file(df: pd.DataFrame, output_path: Path) -> None:
    """Create RecBole .inter file in pre-augmented sequential format."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    header = "user_id:token\titem_id:token\titem_id_list:token_seq"

    with open(output_path, "w") as f:
        f.write(header + "\n")
        for _, row in df.iterrows():
            f.write(
                f"{int(row['user_id'])}\t{int(row['item_id'])}\t{row['item_id_list']}\n"
            )

    print(f"Saved {len(df)} rows to {output_path}")

In [ ]:
RECBOLE_SEQ_PATH.mkdir(parents=True, exist_ok=True)

create_sequential_inter_file(train_seq_df, RECBOLE_SEQ_PATH / "redial_seq.train.inter")
create_sequential_inter_file(valid_seq_df, RECBOLE_SEQ_PATH / "redial_seq.valid.inter")
create_sequential_inter_file(test_seq_df, RECBOLE_SEQ_PATH / "redial_seq.test.inter")

In [ ]:
def create_item_file(movies_df: pd.DataFrame, output_path: Path) -> None:
    """Create RecBole .item file with movie metadata."""
    header = "item_id:token\ttitle:token_seq"

    with open(output_path, "w") as f:
        f.write(header + "\n")
        for _, row in movies_df.iterrows():
            title = str(row["title_norm"]).replace("\t", " ").replace("\n", " ")
            f.write(f"{row['id']}\t{title}\n")

    print(f"Saved {len(movies_df)} items to {output_path}")


create_item_file(movies_df, RECBOLE_SEQ_PATH / "redial_seq.item")

## Save Evaluation Targets and Mappings

In [ ]:
# Save evaluation targets
targets_data = {
    "train_targets": {str(k): v for k, v in train_targets.items()},
    "valid_targets": {str(k): v for k, v in valid_targets.items()},
    "test_targets": {str(k): v for k, v in test_targets.items()},
    "train_user_ids": train_user_ids,
    "valid_user_ids": valid_user_ids,
    "test_user_ids": test_user_ids,
}

# Save dialogue_id → user_id mapping for content-based matching
test_dialogue_to_user = {}
for split_name, dialogues in [
    ("train", train_dialogues_split),
    ("valid", valid_dialogues),
    ("test", test_dialogues),
]:
    for dialogue in dialogues:
        liked_ids = [
            match_title_to_id(t, title_to_id)
            for t in dialogue.get("user_liked", [])
            if match_title_to_id(t, title_to_id) is not None
        ]
        accepted_ids = [
            match_title_to_id(t, title_to_id)
            for t in dialogue.get("recommended_accepted", [])
            if match_title_to_id(t, title_to_id) is not None
        ]
        if not liked_ids or not accepted_ids:
            continue
        did = str(dialogue.get("dialogue_id", ""))
        if split_name == "test" and did:
            uid = test_user_ids[len(test_dialogue_to_user)]
            test_dialogue_to_user[did] = uid

targets_data["test_dialogue_to_user"] = test_dialogue_to_user
print(f"Built dialogue_id mapping for {len(test_dialogue_to_user)} test users")

with open(RECBOLE_SEQ_PATH / "evaluation_targets.json", "w") as f:
    json.dump(targets_data, f, indent=2)

print("Saved evaluation targets:")
print(f"  Train: {len(train_targets)} users")
print(f"  Valid: {len(valid_targets)} users")
print(f"  Test: {len(test_targets)} users")

In [ ]:
# Save ID to title mapping
with open(RECBOLE_SEQ_PATH / "id_to_title.json", "w") as f:
    json.dump({str(k): v for k, v in id_to_title.items()}, f, indent=2)

print(f"Saved {len(id_to_title)} item ID to title mappings")

## Verification

In [ ]:
# Verify file format
for fname in [
    "redial_seq.train.inter",
    "redial_seq.valid.inter",
    "redial_seq.test.inter",
]:
    print(f"\nSample from {fname}:")
    with open(RECBOLE_SEQ_PATH / fname) as f:
        for i, line in enumerate(f):
            print(f"  {line.strip()}")
            if i >= 3:
                break

In [ ]:
# Verify disjoint user sets
train_set = set(train_user_ids)
valid_set = set(valid_user_ids)
test_set = set(test_user_ids)

print("User split verification:")
print(f"  Train users: {len(train_set)}")
print(f"  Valid users: {len(valid_set)}")
print(f"  Test users:  {len(test_set)}")
print(f"  Train-Valid overlap: {len(train_set & valid_set)} (should be 0)")
print(f"  Train-Test overlap:  {len(train_set & test_set)} (should be 0)")
print(f"  Valid-Test overlap:  {len(valid_set & test_set)} (should be 0)")

assert len(train_set & valid_set) == 0, "Train-Valid overlap!"
assert len(train_set & test_set) == 0, "Train-Test overlap!"
assert len(valid_set & test_set) == 0, "Valid-Test overlap!"

In [ ]:
# Verify sequence length stats
for split_name, df in [
    ("train", train_seq_df),
    ("valid", valid_seq_df),
    ("test", test_seq_df),
]:
    seq_lengths = df["item_id_list"].str.split().apply(len)
    print(
        f"{split_name}: min_seq={seq_lengths.min()}, max_seq={seq_lengths.max()}, "
        f"mean_seq={seq_lengths.mean():.1f}, median_seq={seq_lengths.median():.0f}"
    )
    assert seq_lengths.max() <= MAX_ITEM_LIST_LENGTH, (
        "Sequence exceeds MAX_ITEM_LIST_LENGTH!"
    )
    assert seq_lengths.min() >= 1, "Empty sequence found!"

In [ ]:
# Cross-reference with CF data (valid/test user IDs should match exactly)
cf_targets_path = DATA_PATH / "recbole" / "redial" / "evaluation_targets.json"
if cf_targets_path.exists():
    with open(cf_targets_path) as f:
        cf_targets = json.load(f)

    cf_valid_uids = set(cf_targets["valid_user_ids"])
    cf_test_uids = set(cf_targets["test_user_ids"])

    print("Cross-reference with CF data (notebook 06):")
    print(
        f"  Valid users - Sequential: {len(valid_set)}, CF: {len(cf_valid_uids)}, "
        f"Match: {valid_set == cf_valid_uids}"
    )
    print(
        f"  Test users  - Sequential: {len(test_set)}, CF: {len(cf_test_uids)}, "
        f"Match: {test_set == cf_test_uids}"
    )

    # Also verify targets match
    cf_test_targets = {int(k): v for k, v in cf_targets["test_targets"].items()}
    targets_match = all(
        test_targets.get(uid) == cf_test_targets.get(uid) for uid in test_set
    )
    print(f"  Test targets match: {targets_match}")
else:
    print("CF evaluation_targets.json not found, skipping cross-reference.")

In [ ]:
# Verify all item IDs exist in catalog
all_seq_items = set()
for df in [train_seq_df, valid_seq_df, test_seq_df]:
    all_seq_items.update(df["item_id"].unique())
    for seq in df["item_id_list"]:
        all_seq_items.update(int(x) for x in seq.split())

# Remove padding token (0) — used for empty history in anchor rows
all_seq_items.discard(0)

valid_item_ids = set(movies_df["id"])
invalid_items = all_seq_items - valid_item_ids
print(
    f"Total unique items in sequential data: {len(all_seq_items)} (excluding padding)"
)
print(f"Items not in movie catalog: {len(invalid_items)} (should be 0)")
assert len(invalid_items) == 0, f"Invalid items found: {invalid_items}"

## Recommended RecBole Config

Use this config in notebooks 10 (HPO) and 11 (evaluation):

In [ ]:
print("Recommended RecBole config for sequential models:")
print(
    json.dumps(
        {
            "data_path": str(DATA_PATH / "recbole"),
            "dataset": "redial_seq",
            "benchmark_filename": ["train", "valid", "test"],
            "USER_ID_FIELD": "user_id",
            "ITEM_ID_FIELD": "item_id",
            "LIST_SUFFIX": "_list",
            "ITEM_LIST_LENGTH_FIELD": "item_length",
            "MAX_ITEM_LIST_LENGTH": MAX_ITEM_LIST_LENGTH,
            "load_col": {"inter": ["user_id", "item_id", "item_id_list"]},
            "alias_of_item_id": ["item_id_list"],
            "repeatable": True,
            "eval_args": {
                "group_by": "user",
                "order": "TO",
                "split": {"LS": "valid_and_test"},
                "mode": "full",
            },
        },
        indent=2,
    )
)

## Summary

In [ ]:
print("=" * 60)
print("SEQUENTIAL DATA PREPARATION SUMMARY")
print("=" * 60)

print("\nDataset: redial_seq")
print(f"Output directory: {RECBOLE_SEQ_PATH}")

print(
    f"\nTotal users: {len(train_user_ids) + len(valid_user_ids) + len(test_user_ids)}"
)
print(f"  Train: {len(train_user_ids)}")
print(f"  Valid: {len(valid_user_ids)}")
print(f"  Test:  {len(test_user_ids)}")

print("\nInteraction rows (pre-augmented):")
print(f"  train.inter: {len(train_seq_df)} rows")
print(f"  valid.inter: {len(valid_seq_df)} rows")
print(f"  test.inter:  {len(test_seq_df)} rows")

print("\nFiles created:")
for f in sorted(RECBOLE_SEQ_PATH.iterdir()):
    print(f"  - {f.name}")

print("\nData preparation complete! Ready for sequential HPO (notebook 10).")